# Chess Bot — Training on Colab (GPU)

- `src/` (board_utils.py, dataset.py, model.py)
- `data/processed/positions.npy`
- `checkpoints/*.pt` (the models to evaluate)
- `eval/` (eval_new.py, prepare_eval.py, scrape_eval_data.py)
- `requirements.txt`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/Chess_bot')
print("Working directory:", os.getcwd())
print("Files:", os.listdir('.'))    


In [ ]:
!pip install python-chess==1.999 flask==3.0.0 tqdm zstandard requests -q --upgrade
print("Dependencies installed!")

In [ ]:
import torch 
print(f"Pytorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available(): 
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    
# We expect CUDA available: True, GPU: Tesla T4, VRAM: 15.8 GB     

In [ ]:
import requests
import os
import zipfile
from tqdm import tqdm

URL = "https://database.nikonoel.fr/lichess_elite_2023-01.zip"
ZIP_PATH = "data/lichess_elite_2023-01.zip"
PGN_PATH = "data/lichess_elite_2023-01.pgn"

os.makedirs("data", exist_ok=True)

if not os.path.exists(PGN_PATH):
    if not os.path.exists(ZIP_PATH):
        print(f"Downloading {URL} ...")
        response = requests.get(URL, stream=True)
        total = int(response.headers.get('content-length', 0))
        with open(ZIP_PATH, 'wb') as f, tqdm(total=total, unit='B', unit_scale=True) as bar:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
                bar.update(len(chunk))
        print("Download complete.")
    else:
        print(f"ZIP already exists: {ZIP_PATH}")

    print("Extracting...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall("data")
        print(f"Extracted: {zf.namelist()}")
else:
    print(f"PGN already exists: {PGN_PATH}")

print("Done! PGN file ready.")

In [ ]:
import sys
sys.path.insert(0, 'src')

from dataset import parse_pgn_to_arrays, save_array

# Boards are stored as uint8 (4x smaller RAM). Colab free has 12.7 GB RAM:
#   ~85 positions/game, ~1 KB/position => 40,000 games ~ 3.7 GB (safe).
#   80,000 games ~ 7.4 GB and still OOMs at load + concat peak; use <= 40,000.
MAX_GAMES = 40_000

positions, moves, values = parse_pgn_to_arrays(
    "data/lichess_elite_2023-01.pgn",
    max_games=MAX_GAMES,
    min_elo=2000
)
save_array(positions, moves, values, output_dir='data/processed')
print(f"Dataset: {len(positions):,} positions ready for training")

In [ ]:
import subprocess
result = subprocess.run(['python', 'src/train.py'], capture_output=False)

In [ ]:
# Cell 7: Model is already in Drive since we're running from Drive.
# But let's verify and print the path:
import os
best_path = 'checkpoints/best_model.pt'
if os.path.exists(best_path):
    size_mb = os.path.getsize(best_path) / 1e6
    print(f"Best model: {best_path}  ({size_mb:.1f} MB)")
    print(f"Full path:  /content/drive/MyDrive/Chess_bot/{best_path}")
else:
    print("best_model.pt not found — did training complete?")